# TMDB Movie Data Analysis with Spark

Pipeline: fetch movie data from the TMDB API → clean it with PySpark → calculate KPIs → visualise.
Helper functions live in `src/`; this notebook runs them in order and shows the results.

In [1]:
import logging
import os
import sys
from pathlib import Path

# Make the project root importable, so "from src import ..." works from /notebooks
PROJECT_ROOT = Path.cwd() if (Path.cwd() / "requirements.txt").exists() else Path.cwd().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

# Show log messages in the notebook output (stdout = normal, not red, output)
logging.basicConfig(level=logging.INFO, stream=sys.stdout,
                    format="%(asctime)s %(levelname)s %(name)s: %(message)s", force=True)
logging.getLogger("py4j").setLevel(logging.WARNING)   # hide Spark's internal connection messages
logger = logging.getLogger("notebook")

from src import config
from src.fetch import fetch_all_movies, save_raw_json

print("Project root:", PROJECT_ROOT)

Project root: C:\Users\User\Desktop\LABS_Docs\tmdb-spark-analysis


## 1. Fetch movie data from the TMDB API
Raw responses (including cast and crew via `append_to_response=credits`) are saved to
`data/raw/movies.json`, so the API is only called when needed.

In [2]:
FETCH_FROM_API = True   # True = call TMDB now; False = reuse data/raw/movies.json

if FETCH_FROM_API or not config.RAW_MOVIES_FILE.exists():
    save_raw_json(fetch_all_movies())
else:
    logger.info("Using saved raw data: %s", config.RAW_MOVIES_FILE)

2026-11-04 10:28:25,607 WARNING src.fetch: Movie 0: not found (404) - skipped
2026-11-04 10:28:25,959 INFO src.fetch: Movie 299534: fetched 'Avengers: Endgame'
2026-11-04 10:28:26,303 INFO src.fetch: Movie 19995: fetched 'Avatar'
2026-11-04 10:28:26,594 INFO src.fetch: Movie 140607: fetched 'Star Wars: The Force Awakens'
2026-11-04 10:28:26,934 INFO src.fetch: Movie 299536: fetched 'Avengers: Infinity War'
2026-11-04 10:28:27,225 INFO src.fetch: Movie 597: fetched 'Titanic'
2026-11-04 10:28:27,519 INFO src.fetch: Movie 135397: fetched 'Jurassic World'
2026-11-04 10:28:27,784 INFO src.fetch: Movie 420818: fetched 'The Lion King'
2026-11-04 10:28:28,109 INFO src.fetch: Movie 24428: fetched 'The Avengers'
2026-11-04 10:28:28,406 INFO src.fetch: Movie 168259: fetched 'Furious 7'
2026-11-04 10:28:28,712 INFO src.fetch: Movie 99861: fetched 'Avengers: Age of Ultron'
2026-11-04 10:28:29,016 INFO src.fetch: Movie 284054: fetched 'Black Panther'
2026-11-04 10:28:29,305 INFO src.fetch: Movie 124

In [3]:
from pyspark.sql import SparkSession

# Spark's Python workers must use the same Python as this notebook (not the Store "python3")
os.environ["PYSPARK_PYTHON"] = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable

spark = (
    SparkSession.builder
    .appName("tmdb-analysis")
    .master("local[*]")                              # run on this PC, all CPU cores
    .config("spark.sql.shuffle.partitions", "4")     # default is 200: far too many for 18 rows
    .config("spark.sql.session.timeZone", "UTC")     # consistent dates on any PC
    .getOrCreate()
)
spark.sparkContext.setLogLevel("ERROR")              # hide INFO/WARN noise in the notebook

print("Spark", spark.version, "| shuffle partitions:", spark.conf.get("spark.sql.shuffle.partitions"))

Spark 4.3.0 | shuffle partitions: 4


In [4]:
raw_df = spark.read.json(config.RAW_MOVIES_FILE.as_posix(), multiLine=True)

print(f"Rows: {raw_df.count()} | Columns: {len(raw_df.columns)}")
raw_df.select("id", "title", "budget", "revenue").show(truncate=False)

Rows: 18 | Columns: 28
+------+--------------------------------------------+---------+----------+
|id    |title                                       |budget   |revenue   |
+------+--------------------------------------------+---------+----------+
|299534|Avengers: Endgame                           |356000000|2942186933|
|19995 |Avatar                                      |237000000|2923706026|
|140607|Star Wars: The Force Awakens                |245000000|2068223624|
|299536|Avengers: Infinity War                      |300000000|2052415039|
|597   |Titanic                                     |200000000|2264162353|
|135397|Jurassic World                              |150000000|1671537444|
|420818|The Lion King                               |260000000|1662020819|
|24428 |The Avengers                                |220000000|1518815515|
|168259|Furious 7                                   |190000000|1515400000|
|99861 |Avengers: Age of Ultron                     |235000000|1405403694|
|2

### 1.1 Inspect the raw data
Check the columns, data types and nested (JSON-like) fields before cleaning.

In [5]:
print(sorted(raw_df.columns))

['adult', 'backdrop_path', 'belongs_to_collection', 'budget', 'credits', 'genres', 'homepage', 'id', 'imdb_id', 'origin_country', 'original_language', 'original_title', 'overview', 'popularity', 'poster_path', 'production_companies', 'production_countries', 'release_date', 'revenue', 'runtime', 'softcore', 'spoken_languages', 'status', 'tagline', 'title', 'video', 'vote_average', 'vote_count']


In [6]:
raw_df.printSchema(level=2)   # level=2 hides the very long cast/crew details

root
 |-- adult: boolean (nullable = true)
 |-- backdrop_path: string (nullable = true)
 |-- belongs_to_collection: struct (nullable = true)
 |    |-- backdrop_path: string (nullable = true)
 |    |-- id: long (nullable = true)
 |    |-- name: string (nullable = true)
 |    |-- poster_path: string (nullable = true)
 |-- budget: long (nullable = true)
 |-- credits: struct (nullable = true)
 |    |-- cast: array (nullable = true)
 |    |-- crew: array (nullable = true)
 |-- genres: array (nullable = true)
 |    |-- element: struct (containsNull = true)
 |-- homepage: string (nullable = true)
 |-- id: long (nullable = true)
 |-- imdb_id: string (nullable = true)
 |-- origin_country: array (nullable = true)
 |    |-- element: string (containsNull = true)
 |-- original_language: string (nullable = true)
 |-- original_title: string (nullable = true)
 |-- overview: string (nullable = true)
 |-- popularity: double (nullable = true)
 |-- poster_path: string (nullable = true)
 |-- production_com

In [7]:
nested_cols = [field.name for field in raw_df.schema.fields
               if field.dataType.typeName() in ("struct", "array")]
print("Nested columns:", nested_cols)

Nested columns: ['belongs_to_collection', 'credits', 'genres', 'origin_country', 'production_companies', 'production_countries', 'spoken_languages']


In [8]:
from pyspark.sql import functions as F

raw_df.select(
    "title",
    F.col("belongs_to_collection.name").alias("collection"),
    F.col("genres.name").alias("genre_names"),
    F.size("credits.cast").alias("cast_size"),
    F.size("credits.crew").alias("crew_size"),
).show(truncate=False)

+--------------------------------------------+-----------------------------------+-----------------------------------------------+---------+---------+
|title                                       |collection                         |genre_names                                    |cast_size|crew_size|
+--------------------------------------------+-----------------------------------+-----------------------------------------------+---------+---------+
|Avengers: Endgame                           |The Avengers Collection            |[Adventure, Science Fiction, Action]           |103      |614      |
|Avatar                                      |Avatar Collection                  |[Science Fiction, Action, Adventure]           |68       |999      |
|Star Wars: The Force Awakens                |Star Wars Collection               |[Adventure, Action, Science Fiction]           |99       |270      |
|Avengers: Infinity War                      |The Avengers Collection            |[Adventure, 

In [9]:
to_drop = ["adult", "imdb_id", "original_title", "video", "homepage"]
json_like = ["belongs_to_collection", "genres", "production_countries",
             "production_companies", "spoken_languages"]

missing = [c for c in to_drop + json_like + ["credits"] if c not in raw_df.columns]
print("Missing from raw_df:", missing)

Missing from raw_df: []


## 2. Data cleaning and preprocessing
Cleaning functions live in `src/cleaning.py`. Each step returns a new DataFrame; `raw_df` is kept unchanged.

### 2.1 Drop irrelevant columns

In [12]:
from src import cleaning

df = cleaning.drop_irrelevant_columns(raw_df)

print(f"Columns: {len(raw_df.columns)} -> {len(df.columns)}")
print("Dropped:", sorted(set(raw_df.columns) - set(df.columns)))

Columns: 28 -> 23
Dropped: ['adult', 'homepage', 'imdb_id', 'original_title', 'video']
